# 🦉 Kiểm tra và Xác thực Ánh xạ Loài RFCx trực tiếp từ ZIP trên Google Drive (BioListen VN)

Notebook này được thiết kế để kiểm chứng xem bảng ánh xạ loài (`s0` đến `s23`) có chuẩn xác hay không bằng cách:
1. Đọc trực tiếp file metadata `train_tp.csv` và các file âm thanh `.flac` từ tệp ZIP `rfcx-species-audio-detection.zip` trên Google Drive.
2. **Yêu cầu quan trọng:** Không giải nén toàn bộ tệp ZIP ra đĩa ảo `/content/` của Google Colab để tránh bị hết dung lượng (Disk space limit). Chúng ta sử dụng thư viện `zipfile` và `soundfile` để trích xuất file nhị phân vào RAM (in-memory) và phát các file âm thanh được chỉ định.
3. **Tính linh hoạt:** Nếu không tìm thấy file ZIP trên Drive, notebook sẽ tự động kích hoạt chế độ dự phòng (fallback) để stream âm thanh trực tiếp từ thư viện bioacoustic **Xeno-canto** qua internet.

In [ ]:
# Tránh lỗi hiển thị ký tự đặc biệt trên console Colab
!pip install -q soundfile requests pandas

from google.colab import drive
import os

# 1. Kết nối Google Drive
if not os.path.exists('/content/drive'):
    print("Đang kết nối đến Google Drive...")
    drive.mount('/content/drive')
else:
    print("Google Drive đã được kết nối [OK]")

In [ ]:
import zipfile
import pandas as pd
import io
import soundfile as sf
import requests
from IPython.display import display, HTML, Audio

# 2. Đường dẫn tệp ZIP trên Google Drive của bạn
ZIP_PATH = '/content/drive/MyDrive/Datasets/BioListenVN/raw_zips/rfcx-species-audio-detection.zip'

# Từ điển ánh xạ 24 loài gốc vùng Puerto Rico từ s0 -> s23
RFCX_SPECIES_MAP = {
    0: {"id": "s0", "scientific": "Eleutherodactylus coqui", "english": "Common Coqui", "vietnamese": "Ếch Coquí thông thường", "type": "Ếch (Frog)"},
    1: {"id": "s1", "scientific": "Megascops nudipes", "english": "Puerto Rican Screech-Owl", "vietnamese": "Cú mèo Puerto Rico", "type": "Chim (Bird)"},
    2: {"id": "s2", "scientific": "Todus mexicanus", "english": "Puerto Rican Tody", "vietnamese": "Chim Tody Puerto Rico", "type": "Chim (Bird)"},
    3: {"id": "s3", "scientific": "Coereba flaveola", "english": "Bananaquit", "vietnamese": "Chim Bananaquit", "type": "Chim (Bird)"},
    4: {"id": "s4", "scientific": "Melanerpes portoricensis", "english": "Puerto Rican Woodpecker", "vietnamese": "Gõ kiến Puerto Rico", "type": "Chim (Bird)"},
    5: {"id": "s5", "scientific": "Loxigilla portoricensis", "english": "Puerto Rican Bullfinch", "vietnamese": "Chim sẻ thông Puerto Rico", "type": "Chim (Bird)"},
    6: {"id": "s6", "scientific": "Vireo latimeri", "english": "Puerto Rican Vireo", "vietnamese": "Chim Vireo Puerto Rico", "type": "Chim (Bird)"},
    7: {"id": "s7", "scientific": "Spindalis portoricensis", "english": "Puerto Rican Spindalis", "vietnamese": "Chim Spindalis Puerto Rico", "type": "Chim (Bird)"},
    8: {"id": "s8", "scientific": "Crotophaga ani", "english": "Smooth-billed Ani", "vietnamese": "Chim Ani mỏ nhẵn", "type": "Chim (Bird)"},
    9: {"id": "s9", "scientific": "Buteo platypterus", "english": "Broad-winged Hawk", "vietnamese": "Diều hâu cánh rộng", "type": "Chim (Bird)"},
    10: {"id": "s10", "scientific": "Leptodactylus albilabris", "english": "White-lipped Frog", "vietnamese": "Ếch môi trắng", "type": "Ếch (Frog)"},
    11: {"id": "s11", "scientific": "Eleutherodactylus antillensis", "english": "Red-eyed Coqui", "vietnamese": "Ếch mắt đỏ", "type": "Ếch (Frog)"},
    12: {"id": "s12", "scientific": "Eleutherodactylus brittoni", "english": "Britton's Coqui", "vietnamese": "Ếch Britton", "type": "Ếch (Frog)"},
    13: {"id": "s13", "scientific": "Eleutherodactylus wightmanae", "english": "Wightman's Coqui", "vietnamese": "Ếch Wightman", "type": "Ếch (Frog)"},
    14: {"id": "s14", "scientific": "Eleutherodactylus richmondi", "english": "Richmond's Coqui", "vietnamese": "Ếch Richmond", "type": "Ếch (Frog)"},
    15: {"id": "s15", "scientific": "Eleutherodactylus gryllus", "english": "Cricket Coqui", "vietnamese": "Ếch dế", "type": "Ếch (Frog)"},
    16: {"id": "s16", "scientific": "Eleutherodactylus locustus", "english": "Locust Coqui", "vietnamese": "Ếch châu chấu", "type": "Ếch (Frog)"},
    17: {"id": "s17", "scientific": "Eleutherodactylus hedricki", "english": "Hedrick's Coqui", "vietnamese": "Ếch Hedrick", "type": "Ếch (Frog)"},
    18: {"id": "s18", "scientific": "Eleutherodactylus unicolor", "english": "Bronze Coqui", "vietnamese": "Ếch đồng", "type": "Ếch (Frog)"},
    19: {"id": "s19", "scientific": "Eleutherodactylus portoricensis", "english": "Puerto Rican Coqui", "vietnamese": "Ếch rừng Puerto Rico", "type": "Ếch (Frog)"},
    20: {"id": "s20", "scientific": "Eleutherodactylus cooki", "english": "Guajón Coqui", "vietnamese": "Ếch đá", "type": "Ếch (Frog)"},
    21: {"id": "s21", "scientific": "Eleutherodactylus eneidae", "english": "Eneida's Coqui", "vietnamese": "Ếch Eneida", "type": "Ếch (Frog)"},
    22: {"id": "s22", "scientific": "Eleutherodactylus karlschmidti", "english": "Karlschmidt's Coqui", "vietnamese": "Ếch suối Karlschmidt", "type": "Ếch (Frog)"},
    23: {"id": "s23", "scientific": "Lithobates catesbeianus", "english": "American Bullfrog", "vietnamese": "Ếch ương lớn", "type": "Ếch (Frog)"}
}

df_tp = pd.DataFrame()
has_zip = False

# 3. Đọc train_tp.csv trực tiếp từ file ZIP trên Drive
if os.path.exists(ZIP_PATH):
    print(f"Đang mở tệp ZIP trên Drive: {ZIP_PATH}")
    try:
        with zipfile.ZipFile(ZIP_PATH, 'r') as z:
            all_files = z.namelist()
            tp_csv_member = [f for f in all_files if f.endswith('train_tp.csv')]
            if tp_csv_member:
                with z.open(tp_csv_member[0]) as f:
                    df_tp = pd.read_csv(f)
                has_zip = True
                print(f"[OK] Đã nạp metadata train_tp.csv từ ZIP (Số dòng: {len(df_tp)})")
            else:
                print("[Cảnh báo] Không tìm thấy train_tp.csv trong ZIP.")
    except Exception as e:
        print(f"[Lỗi] Không thể đọc file ZIP: {e}")
else:
    print(f"[Lưu ý] Không tìm thấy file ZIP tại '{ZIP_PATH}'.")
    print("Chuyển sang chế độ dự phòng (Stream trực tiếp từ thư viện Xeno-canto).")

### Các hàm Helper xử lý API và Nhạc

In [ ]:
def get_xeno_canto_audio_url(scientific_name):
    """
    Truy vấn API Xeno-canto để lấy link stream audio của loài.
    """
    api_url = f"https://xeno-canto.org/api/2/recordings?query={scientific_name.replace(' ', '%20')}"
    try:
        resp = requests.get(api_url, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            recordings = data.get('recordings', [])
            if recordings:
                file_url = recordings[0].get('file')
                if file_url:
                    if file_url.startswith('//'):
                        file_url = 'https:' + file_url
                    return file_url
    except Exception as e:
        print(f"Lỗi API Xeno-canto: {e}")
    return None

def play_from_xeno_canto(info):
    """
    Stream âm thanh từ Xeno-canto trực tuyến.
    """
    print(f"-> Đang stream thử âm thanh từ thư viện Xeno-canto trực tuyến...")
    audio_url = get_xeno_canto_audio_url(info['scientific'])
    if audio_url:
        print(f"Link audio: {audio_url}")
        display(Audio(url=audio_url))
    else:
        print("❌ Không tìm thấy bản ghi nào trên Xeno-canto.")

In [ ]:
def play_species(id_str, sample_idx=0):
    """
    Trình phát và kiểm chứng tổng hợp:
    Ưu tiên đọc file FLAC trong ZIP trên Google Drive. Nếu không có ZIP, sẽ fallback stream từ Xeno-canto.
    """
    id_str = id_str.strip().lower()
    target_key = None
    for k, v in RFCX_SPECIES_MAP.items():
        if v['id'] == id_str:
            target_key = k
            break
            
    if target_key is None:
        try:
            target_key = int(id_str)
        except ValueError:
            pass
            
    if target_key not in RFCX_SPECIES_MAP:
        print(f"❌ Không tìm thấy loài với nhãn '{id_str}' (Vui lòng nhập s0 -> s23)")
        return
        
    info = RFCX_SPECIES_MAP[target_key]
    
    # Hiển thị bảng thông tin sinh học
    html_template = f"""
    <div style='border: 2px solid #2E7D32; border-radius: 8px; padding: 15px; background-color: #f1f8e9; max-width: 650px; font-family: Arial, sans-serif;'>
        <h3 style='margin-top: 0; color: #2E7D32;'>🔎 HỒ SƠ LOÀI: {info['id'].upper()}</h3>
        <table style='width: 100%; border-collapse: collapse;'>
            <tr style='border-bottom: 1px solid #ddd;'>
                <td style='padding: 6px 0; font-weight: bold; width: 35%;'>Tên khoa học:</td>
                <td style='padding: 6px 0; font-style: italic;'>{info['scientific']}</td>
            </tr>
            <tr style='border-bottom: 1px solid #ddd;'>
                <td style='padding: 6px 0; font-weight: bold;'>Phân nhóm:</td>
                <td style='padding: 6px 0;'>{info['type']}</td>
            </tr>
            <tr style='border-bottom: 1px solid #ddd;'>
                <td style='padding: 6px 0; font-weight: bold;'>Tên tiếng Anh:</td>
                <td style='padding: 6px 0;'>{info['english']}</td>
            </tr>
            <tr>
                <td style='padding: 6px 0; font-weight: bold;'>Tên tiếng Việt đề xuất:</td>
                <td style='padding: 6px 0; color: #D32F2F; font-weight: bold;'>{info['vietnamese']}</td>
            </tr>
        </table>
    </div>
    """
    display(HTML(html_template))
    
    # Nếu có tệp ZIP trên Drive và đọc được metadata
    if has_zip and not df_tp.empty:
        species_records = df_tp[df_tp['species_id'] == target_key]
        if species_records.empty:
            print(f"Không tìm thấy bản ghi True Positive cho loài này trong train_tp.csv.")
            play_from_xeno_canto(info)
            return
            
        sample_idx = min(sample_idx, len(species_records) - 1)
        record = species_records.iloc[sample_idx]
        recording_id = record['recording_id']
        t_min, t_max = record['t_min'], record['t_max']
        
        print(f"📂 Tìm thấy mẫu trong ZIP: File {recording_id}.flac (Đoạn ghi âm kêu: {t_min:.1f}s - {t_max:.1f}s)")
        print("-> Đang đọc nhị phân giải mã âm thanh trực tiếp vào RAM (không ghi xuống đĩa Colab)... ")
        
        try:
            with zipfile.ZipFile(ZIP_PATH, 'r') as z:
                flac_member = [f for f in z.namelist() if f.endswith(f"{recording_id}.flac")]
                if not flac_member:
                    print(f"❌ Warning: Không tìm thấy file {recording_id}.flac trong ZIP.")
                    play_from_xeno_canto(info)
                    return
                    
                with z.open(flac_member[0]) as f_audio:
                    # Đọc nhị phân trực tiếp bằng soundfile và BytesIO
                    data, sr = sf.read(io.BytesIO(f_audio.read()))
                    
                    # Cắt đoạn trích 6 giây xung quanh tiếng kêu để kiểm chứng dễ nhất
                    t_center = (t_min + t_max) / 2
                    start_sample = int(max(0, (t_center - 3.0) * sr))
                    end_sample = int(min(len(data), (t_center + 3.0) * sr))
                    clip_data = data[start_sample:end_sample]
                    
                    print(f"🔊 TRÌNH PHÁT ĐOẠN CẮT TIẾNG KÊU ({t_min:.1f}s - {t_max:.1f}s):")
                    display(Audio(data=clip_data, rate=sr))
                    
                    print("🔊 TRÌNH PHÁT TOÀN BỘ FILE 60 GIÂY:")
                    display(Audio(data=data, rate=sr))
        except Exception as e:
            print(f"❌ Lỗi giải mã file ZIP: {e}. Chuyển sang stream từ Xeno-canto...")
            play_from_xeno_canto(info)
    else:
        # Chế độ dự phòng
        play_from_xeno_canto(info)

## 🎧 PHẦN CHẠY THỬ & XÁC THỰC ÂM THANH

Bấm chạy các Cell bên dưới để kiểm chứng âm thanh. Bạn có thể thay đổi nhãn (từ `'s0'` đến `'s23'`) và tham số `sample_idx` để nghe các file ghi âm thực địa khác nhau trong bộ dữ liệu của bạn.

In [ ]:
# 1. Nghe tiếng ếch s0 (Common Coqui) - Tiếng ếch kêu "co-qui..." đặc trưng
play_species('s0', sample_idx=0)

In [ ]:
# 2. Nghe tiếng cú mèo s1 (Puerto Rican Screech-Owl) - Chim kêu dồn dập về đêm
play_species('s1', sample_idx=0)

In [ ]:
# 3. Nghe tiếng chim Tody s2 (Puerto Rican Tody) - Tiếng chim líu lo rực rỡ vùng nhiệt đới
play_species('s2', sample_idx=0)

In [ ]:
# 4. Nghe tiếng ếch s10 (White-lipped Frog) - Tiếng ếch kêu trầm đục rải rác
play_species('s10', sample_idx=0)

In [ ]:
# Chạy cell này để in nhanh checklist toàn bộ 24 loài để dễ đối chiếu
print("=== BẢNG ĐỐI CHIẾU 24 LOÀI ===")
for k, v in RFCX_SPECIES_MAP.items():
    print(f"{v['id'].upper()} -> {v['scientific']} ({v['type']}) | Anh: {v['english']} | Việt: {v['vietnamese']}")